# ДЗ №12

В этом ноутбуке добавляем и сравниваем новые классификаторы:

- `MultinomialNB`
- `LogisticRegression`
- `LinearSVC`
- `SGDClassifier`
- `KNeighborsClassifier`

Эксперименты выполняются на двух датасетах:

1. `emotion`
2. `20_newsgroups(4)`

Для представления текстов используется:

- `CountVectorizer`
- `TfidfVectorizer`

Качество оценивается с помощью:

- `F1 micro`
- `F1 macro`
- `F1 weighted`

Основной вывод делается по `F1 macro`, так как эта метрика одинаково учитывает все классы.


## 1. Импорты

In [1]:
import re
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import f1_score, classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings("ignore")

## 2. Предобработка текста


In [2]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_texts(texts):
    return [clean_text(t) for t in texts]

## 3. Функция для обучения и оценки моделей

In [3]:
def evaluate_classifiers(X_train, X_test, y_train, y_test, representation_name, dataset_name):
    models = {
        "MultinomialNB": MultinomialNB(alpha=1.0),
        "LogisticRegression": LogisticRegression(
            C=1.0,
            max_iter=1000,
            solver="lbfgs",
            n_jobs=-1,
            random_state=42
        ),
        "LinearSVC": LinearSVC(
            C=1.0,
            max_iter=5000,
            random_state=42
        ),
        "SGDClassifier_hinge": SGDClassifier(
            loss="hinge",
            alpha=1e-4,
            max_iter=1000,
            tol=1e-3,
            random_state=42,
            n_jobs=-1
        ),
        "SGDClassifier_log_loss": SGDClassifier(
            loss="log_loss",
            alpha=1e-4,
            max_iter=1000,
            tol=1e-3,
            random_state=42,
            n_jobs=-1
        ),
        "KNeighbors_cosine": KNeighborsClassifier(
            n_neighbors=5,
            metric="cosine",
            algorithm="brute",
            n_jobs=-1
        )
    }

    rows = []

    for model_name, model in models.items():
        print(f"Training {dataset_name} | {representation_name} | {model_name}")
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        rows.append({
            "dataset": dataset_name,
            "representation": representation_name,
            "model": model_name,
            "f1_micro": f1_score(y_test, pred, average="micro"),
            "f1_macro": f1_score(y_test, pred, average="macro"),
            "f1_weighted": f1_score(y_test, pred, average="weighted")
        })

    return pd.DataFrame(rows)

## 4. Dataset 1: emotion

In [4]:
emotion_dataset = load_dataset("emotion")

emotion_train_texts = list(emotion_dataset["train"]["text"])
emotion_test_texts = list(emotion_dataset["test"]["text"])

emotion_train_labels = np.array(emotion_dataset["train"]["label"])
emotion_test_labels = np.array(emotion_dataset["test"]["label"])

emotion_train_clean = preprocess_texts(emotion_train_texts)
emotion_test_clean = preprocess_texts(emotion_test_texts)

print("Emotion train:", len(emotion_train_clean))
print("Emotion test:", len(emotion_test_clean))

Emotion train: 16000
Emotion test: 2000


### 4.1 CountVectorizer для emotion

In [5]:
emotion_count_vectorizer = CountVectorizer(
    max_features=20000,
    min_df=2
)

X_train_emotion_count = emotion_count_vectorizer.fit_transform(emotion_train_clean)
X_test_emotion_count = emotion_count_vectorizer.transform(emotion_test_clean)

print(X_train_emotion_count.shape, X_test_emotion_count.shape)

(16000, 7258) (2000, 7258)


### 4.2 TfidfVectorizer для emotion

In [6]:
emotion_tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    min_df=2
)

X_train_emotion_tfidf = emotion_tfidf_vectorizer.fit_transform(emotion_train_clean)
X_test_emotion_tfidf = emotion_tfidf_vectorizer.transform(emotion_test_clean)

print(X_train_emotion_tfidf.shape, X_test_emotion_tfidf.shape)

(16000, 7258) (2000, 7258)


### 4.3 Обучение новых классификаторов на emotion

In [7]:
emotion_count_results = evaluate_classifiers(
    X_train_emotion_count,
    X_test_emotion_count,
    emotion_train_labels,
    emotion_test_labels,
    representation_name="count",
    dataset_name="emotion"
)

emotion_tfidf_results = evaluate_classifiers(
    X_train_emotion_tfidf,
    X_test_emotion_tfidf,
    emotion_train_labels,
    emotion_test_labels,
    representation_name="tfidf",
    dataset_name="emotion"
)

emotion_results = pd.concat([emotion_count_results, emotion_tfidf_results], ignore_index=True)
emotion_results.sort_values("f1_macro", ascending=False)

Training emotion | count | MultinomialNB
Training emotion | count | LogisticRegression
Training emotion | count | LinearSVC
Training emotion | count | SGDClassifier_hinge
Training emotion | count | SGDClassifier_log_loss
Training emotion | count | KNeighbors_cosine
Training emotion | tfidf | MultinomialNB
Training emotion | tfidf | LogisticRegression
Training emotion | tfidf | LinearSVC
Training emotion | tfidf | SGDClassifier_hinge
Training emotion | tfidf | SGDClassifier_log_loss
Training emotion | tfidf | KNeighbors_cosine


,dataset,representation,model,f1_micro,f1_macro,f1_weighted
8,emotion,tfidf,LinearSVC,0.8900,0.839342,0.889116
2,emotion,count,LinearSVC,0.8860,0.835915,0.886058
9,emotion,tfidf,SGDClassifier_hinge,0.8925,0.833153,0.890394
3,emotion,count,SGDClassifier_hinge,0.8875,0.829658,0.887139
1,emotion,count,LogisticRegression,0.8865,0.829554,0.885396
4,emotion,count,SGDClassifier_log_loss,0.8850,0.826738,0.884636
7,emotion,tfidf,LogisticRegression,0.8655,0.801441,0.860764
10,emotion,tfidf,SGDClassifier_log_loss,0.8380,0.747777,0.828492
0,emotion,count,MultinomialNB,0.8135,0.673934,0.797031
11,emotion,tfidf,KNeighbors_cosine,0.7195,0.642880,0.711426


## 5. Dataset 2: 20_newsgroups(4)


In [8]:
news_dataset = load_dataset("SetFit/20_newsgroups")

categories = [
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.graphics",
    "comp.windows.x"
]

train_news = news_dataset["train"].filter(lambda x: x["label_text"] in categories)
test_news = news_dataset["test"].filter(lambda x: x["label_text"] in categories)

news_train_texts = list(train_news["text"])
news_test_texts = list(test_news["text"])

news_train_label_texts = list(train_news["label_text"])
news_test_label_texts = list(test_news["label_text"])

label2id = {label: i for i, label in enumerate(categories)}
id2label = {i: label for label, i in label2id.items()}

news_train_labels = np.array([label2id[label] for label in news_train_label_texts])
news_test_labels = np.array([label2id[label] for label in news_test_label_texts])

news_train_clean = preprocess_texts(news_train_texts)
news_test_clean = preprocess_texts(news_test_texts)

print("20_newsgroups train:", len(news_train_clean))
print("20_newsgroups test:", len(news_test_clean))
print(label2id)

Using the latest cached version of the dataset since SetFit/20_newsgroups couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\Masha\.cache\huggingface\datasets\SetFit___20_newsgroups\default\0.0.0\f1b91292074e7cfb69be58b642d583ec262f30ed (last modified on Wed Apr 15 13:30:38 2026).


20_newsgroups train: 2345
20_newsgroups test: 1561
{'comp.sys.ibm.pc.hardware': 0, 'comp.sys.mac.hardware': 1, 'comp.graphics': 2, 'comp.windows.x': 3}


### 5.1 CountVectorizer для 20_newsgroups(4)

In [9]:
news_count_vectorizer = CountVectorizer(
    max_features=30000,
    min_df=2
)

X_train_news_count = news_count_vectorizer.fit_transform(news_train_clean)
X_test_news_count = news_count_vectorizer.transform(news_test_clean)

print(X_train_news_count.shape, X_test_news_count.shape)

(2345, 9431) (1561, 9431)


### 5.2 TfidfVectorizer для 20_newsgroups(4)

In [10]:
news_tfidf_vectorizer = TfidfVectorizer(
    max_features=30000,
    min_df=2
)

X_train_news_tfidf = news_tfidf_vectorizer.fit_transform(news_train_clean)
X_test_news_tfidf = news_tfidf_vectorizer.transform(news_test_clean)

print(X_train_news_tfidf.shape, X_test_news_tfidf.shape)

(2345, 9431) (1561, 9431)


### 5.3 Обучение новых классификаторов на 20_newsgroups(4)

In [11]:
news_count_results = evaluate_classifiers(
    X_train_news_count,
    X_test_news_count,
    news_train_labels,
    news_test_labels,
    representation_name="count",
    dataset_name="20_newsgroups(4)"
)

news_tfidf_results = evaluate_classifiers(
    X_train_news_tfidf,
    X_test_news_tfidf,
    news_train_labels,
    news_test_labels,
    representation_name="tfidf",
    dataset_name="20_newsgroups(4)"
)

news_results = pd.concat([news_count_results, news_tfidf_results], ignore_index=True)
news_results.sort_values("f1_macro", ascending=False)

Training 20_newsgroups(4) | count | MultinomialNB
Training 20_newsgroups(4) | count | LogisticRegression
Training 20_newsgroups(4) | count | LinearSVC
Training 20_newsgroups(4) | count | SGDClassifier_hinge
Training 20_newsgroups(4) | count | SGDClassifier_log_loss
Training 20_newsgroups(4) | count | KNeighbors_cosine
Training 20_newsgroups(4) | tfidf | MultinomialNB
Training 20_newsgroups(4) | tfidf | LogisticRegression
Training 20_newsgroups(4) | tfidf | LinearSVC
Training 20_newsgroups(4) | tfidf | SGDClassifier_hinge
Training 20_newsgroups(4) | tfidf | SGDClassifier_log_loss
Training 20_newsgroups(4) | tfidf | KNeighbors_cosine


,dataset,representation,model,f1_micro,f1_macro,f1_weighted
10,20_newsgroups(4),tfidf,SGDClassifier_log_loss,0.766816,0.767952,0.768045
7,20_newsgroups(4),tfidf,LogisticRegression,0.766176,0.767622,0.767771
6,20_newsgroups(4),tfidf,MultinomialNB,0.765535,0.765526,0.765747
8,20_newsgroups(4),tfidf,LinearSVC,0.755926,0.756417,0.756476
9,20_newsgroups(4),tfidf,SGDClassifier_hinge,0.744395,0.745240,0.745311
0,20_newsgroups(4),count,MultinomialNB,0.743113,0.743859,0.743977
4,20_newsgroups(4),count,SGDClassifier_log_loss,0.727098,0.727012,0.727233
3,20_newsgroups(4),count,SGDClassifier_hinge,0.722614,0.722934,0.723102
1,20_newsgroups(4),count,LogisticRegression,0.711083,0.711538,0.711581
2,20_newsgroups(4),count,LinearSVC,0.682896,0.683258,0.683270


## 6. Общая таблица результатов

In [12]:
all_results = pd.concat([emotion_results, news_results], ignore_index=True)
all_results_sorted = all_results.sort_values(["dataset", "f1_macro"], ascending=[True, False])
all_results_sorted

,dataset,representation,model,f1_micro,f1_macro,f1_weighted
22,20_newsgroups(4),tfidf,SGDClassifier_log_loss,0.766816,0.767952,0.768045
19,20_newsgroups(4),tfidf,LogisticRegression,0.766176,0.767622,0.767771
18,20_newsgroups(4),tfidf,MultinomialNB,0.765535,0.765526,0.765747
20,20_newsgroups(4),tfidf,LinearSVC,0.755926,0.756417,0.756476
21,20_newsgroups(4),tfidf,SGDClassifier_hinge,0.744395,0.745240,0.745311
12,20_newsgroups(4),count,MultinomialNB,0.743113,0.743859,0.743977
16,20_newsgroups(4),count,SGDClassifier_log_loss,0.727098,0.727012,0.727233
15,20_newsgroups(4),count,SGDClassifier_hinge,0.722614,0.722934,0.723102
13,20_newsgroups(4),count,LogisticRegression,0.711083,0.711538,0.711581
14,20_newsgroups(4),count,LinearSVC,0.682896,0.683258,0.683270


## 7. Лучшие модели для каждого датасета

In [13]:
best_by_dataset = (
    all_results
    .sort_values("f1_macro", ascending=False)
    .groupby("dataset", as_index=False)
    .first()
)

best_by_dataset

,dataset,representation,model,f1_micro,f1_macro,f1_weighted
0,20_newsgroups(4),tfidf,SGDClassifier_log_loss,0.766816,0.767952,0.768045
1,emotion,tfidf,LinearSVC,0.890000,0.839342,0.889116


## 8. Сравнение моделей внутри каждого представления

In [14]:
pivot_macro = all_results.pivot_table(
    index=["dataset", "model"],
    columns="representation",
    values="f1_macro"
).reset_index()

pivot_macro

representation,dataset,model,count,tfidf
0,20_newsgroups(4),KNeighbors_cosine,0.458093,0.667907
1,20_newsgroups(4),LinearSVC,0.683258,0.756417
2,20_newsgroups(4),LogisticRegression,0.711538,0.767622
3,20_newsgroups(4),MultinomialNB,0.743859,0.765526
4,20_newsgroups(4),SGDClassifier_hinge,0.722934,0.745240
5,20_newsgroups(4),SGDClassifier_log_loss,0.727012,0.767952
6,emotion,KNeighbors_cosine,0.257753,0.642880
7,emotion,LinearSVC,0.835915,0.839342
8,emotion,LogisticRegression,0.829554,0.801441
9,emotion,MultinomialNB,0.673934,0.430369
